In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.common.exceptions import NoSuchElementException
from selenium.webdriver.support import expected_conditions as EC
from datetime import datetime, timedelta
import time
from time import sleep
from contextlib import suppress
import re
from threading import Thread
from  concurrent.futures import ThreadPoolExecutor


#.....................................................................................................................
#scrape male and female players based on Elo rankings
import requests
from bs4 import BeautifulSoup
import csv
from datetime import datetime
import os
import unicodedata
import re

def clean_text(text):
    """
    Clean text by normalizing Unicode characters and removing problematic symbols
    """
    if not text:
        return text
    
    # Normalize Unicode characters (converts accented characters to their base forms)
    text = unicodedata.normalize('NFKD', text)
    
    # Replace non-breaking spaces and other problematic whitespace characters
    text = text.replace('\u00A0', ' ')  # Non-breaking space
    text = text.replace('\u2009', ' ')  # Thin space
    text = text.replace('\u202F', ' ')  # Narrow no-break space
    text = text.replace('\u2007', ' ')  # Figure space
    text = text.replace('\u2008', ' ')  # Punctuation space
    
    # Replace multiple whitespace characters with single space
    text = re.sub(r'\s+', ' ', text)
    
    # Remove any remaining non-ASCII characters that might cause issues
    # But keep accented characters by only removing control characters
    text = ''.join(char for char in text if unicodedata.category(char)[0] != 'C')
    
    return text.strip()

def scrape_tennis_elo_rankings(urls):
    # URL for Tennis Abstract ATP ELO ratings
    
    # Headers to mimic a browser request
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    for url in urls:
        try:
            # Send GET request to the URL
            print(f"Fetching data from {url}...")
            response = requests.get(url, headers=headers)
            
            # Explicitly set encoding to handle special characters properly
            response.encoding = 'utf-8'
            
            # Check if the request was successful
            if response.status_code == 200:
                print(f"Successfully retrieved the page: Status code {response.status_code}")
                soup = BeautifulSoup(response.text, 'html.parser')
                table = soup.find_all('table')
                if not table:
                    print("Could not find the rankings table.")
                    return None
                table = table[2]

                # Extract table headers
                headers_list = []
                header_row = table.find('tr')
                if header_row:
                    headers_list = [clean_text(th.text) for th in header_row.find_all('th')]
                    headers_list = [string for string in headers_list if len(string) > 0]
                
                if not headers_list:
                    print("Could not find table headers.")
                    return None
                
                player_rows = table.find_all('tr')[1:]
                
                if not player_rows:
                    print("Could not find player rows.")
                    return None
                
                print(f"Found {len(player_rows)} player rows")
                
                # Prepare data structure
                rankings_data = []
                
                for num, row in enumerate(player_rows):
                    try:
                        # Extract all cells in the row and clean the text
                        cells = [clean_text(x.text) for x in row.find_all(['td', 'th'])]
                        cells = [string for string in cells if len(string) > 0]
                        
                        if len(cells) >= len(headers_list):
                            player_data = {}
                            for i, header in enumerate(headers_list):
                                player_data[header] = cells[i]
                            
                            rankings_data.append(player_data)
                        else:
                            print(f"Row {num+1} has fewer cells ({len(cells)}) than headers ({len(headers_list)})")
                    except Exception as e:
                        print(f"Error extracting data from row: {e}")
                        continue
                
                
                # Save data to CSV
                base = r"C:\Users\HP\source\repos\Rehoboam\Rehoboam\Data\Tennis"
                atp_file = f"men_elo_rankings.csv"
                wta_file = f"women_elo_rankings.csv"
                csv_filename = os.path.join(base, atp_file) if 'atp' in url else os.path.join(base, wta_file)
                
                # Create directory if it doesn't exist
                os.makedirs(os.path.dirname(csv_filename), exist_ok=True)
                
                with open(csv_filename, 'w', newline='', encoding='utf-8') as csvfile:
                    writer = csv.DictWriter(csvfile, fieldnames=headers_list)
                    writer.writeheader()
                    for player_data in rankings_data:
                        writer.writerow(player_data)
                
                print(f"Data saved to {csv_filename}")
                
            else:
                print(f"Failed to retrieve the page. Status code: {response.status_code}")
                return None
        
        except Exception as e:
            print(f"An error occurred: {e}")
            return None
#........................................................................................................................

def setup_driver():
    options = webdriver.ChromeOptions()
    options.add_argument('--disable-notifications')
    options.add_argument('--headless')  # Run in background
    options.add_argument('--disable-gpu')  # Recommended for headless
    options.add_argument('--window-size=1920,1080')  # Set a standard window size
    options.add_argument('--no-sandbox')  # Bypass OS security model
    options.add_argument('--disable-dev-shm-usage')  # Overcome limited resource problems
    
    # Add a realistic user agent
    options.add_argument('--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36')
    
    # Some additional useful options
    options.add_argument('--disable-blink-features=AutomationControlled')  # Hide automation
    options.add_experimental_option('excludeSwitches', ['enable-automation'])  # Hide automation
    options.add_experimental_option('useAutomationExtension', False)  # Hide automation
    
    driver = webdriver.Chrome(options=options)
    
    # Execute JS to modify navigator.webdriver flag
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
    
    return driver

def get_tournament_name_and_type(header_text):
    """
    Extracts the tournament name and type (e.g., 'ATP', 'WTA', 'ITF') from header text
    """
    try:
        parts = header_text.strip().split('\n')
        if len(parts) > 1:
            tournament_type = parts[0].strip()
            tournament_name = parts[1].strip().replace(':', '')
            return (tournament_name, tournament_type)
        return (header_text.strip(), "") 
    except:
        return (header_text.strip(), "")

def determine_surface_from_text(raw_text):
    """
    Determine surface from tournament header text
    """
    raw_text_lower = raw_text.lower()
    if 'hard' in raw_text_lower:
        return 0  # hard
    elif 'clay' in raw_text_lower:
        return 1  # clay
    elif 'grass' in raw_text_lower:
        return 2  # grass
    else:
        return 0  # default to hard

def is_desired_tournament(match_element):
    try:
        tournament_header = match_element.find_element(By.XPATH, "./preceding::div[contains(@class, 'headerLeague__wrapper')][1]")
        raw_text = tournament_header.text.strip()
        tournament_name, tournament_type = get_tournament_name_and_type(raw_text)
        # Determine surface from raw text
        surface = determine_surface_from_text(raw_text)
        
        desired_tournaments = [
            'ATP',
            'WTA',
            'CHALLENGER MEN'
            # 'CHALLENGER WOMEN'
            # 'ITF Men',
            # 'ITF Women',
            # 'United Cup',
            # 'Davis Cup',
            # 'Billie Jean King Cup'
        ]

        is_desired: bool = False

        # print(f'tournament name = {tournament_name}')
        # print(f'tournament type = {tournament_type}')

        if 'Qualification' not in tournament_type: 
            for tournament in desired_tournaments:
                if tournament in tournament_type and 'DOUBLE' not in tournament_type:
                    print("GOOOOOOOOOOOOOOOOOt HEEEEEEEEEEEEEEEEEEEEre")
                    is_desired = True
                    break

        return (is_desired, tournament_name, tournament_type, surface)
        
    except NoSuchElementException:
        return (False, "", "", 0)

def get_upcoming_matches(driver, day=0):
    driver.get("https://www.flashscore.com/tennis/")
    upcoming = []

    with suppress(Exception):
        accept_button = WebDriverWait(driver, 5).until(
            EC.element_to_be_clickable((By.ID, "onetrust-accept-btn-handler"))
        )
        accept_button.click()
    
    if day > 0:
        for _ in range(day):
            next = WebDriverWait(driver, 10).until(
                EC.element_to_be_clickable((By.CSS_SELECTOR, "button[data-day-picker-arrow='next']"))
            )
            next = driver.find_element(By.CSS_SELECTOR, "button[data-day-picker-arrow='next']")
            driver.execute_script("arguments[0].click();", next)
            sleep(3)
    

    try:
        WebDriverWait(driver, 10).until(
            EC.presence_of_all_elements_located((By.CLASS_NAME, "event__match"))
        )
        
        matches = driver.find_elements(By.CLASS_NAME, "event__match")

        for match in matches:
            try:
                is_tournament, tournament_name, tournament_type, surface = is_desired_tournament(match)
                if is_tournament:
                    players = match.find_elements(By.CLASS_NAME, "event__participant")
                    time = match.find_element(By.CLASS_NAME, "event__time")
                    match_link = match.find_element(By.CLASS_NAME, "eventRowLink").get_attribute("href")
                    
                    upcoming.append({
                        'tournament': tournament_name,
                        'type': tournament_type,
                        'player1': players[0].text,
                        'player2': players[1].text,
                        'time': time.text[:5],
                        'link': match_link,
                        'surface': surface
                    })
            except Exception as e:
                continue

    except Exception as e:
        print(f"Error getting upcoming matches: {e}")

    return upcoming

def scrape_upcoming_games():
    day = 0 # 0 for today, 1 for next day matches
    
    driver = setup_driver()
    try:
        upcoming = get_upcoming_matches(driver, day)
        number_of_matches = len(upcoming)

        file1 = r"C:\Users\HP\source\repos\Rehoboam\Rehoboam\Data\Tennis\ATP_Singles_Matches.txt"
        file2 = r"C:\Users\HP\source\repos\Rehoboam\Rehoboam\Data\Tennis\WTA_Singles_Matches.txt"
        file3 = r"C:\Users\HP\source\repos\Rehoboam\Rehoboam\Data\Tennis\Challenger_singles.txt"

        
        last_saved = 0 #Default value is 0
        for number, match in enumerate(upcoming):
            # Convert surface number to string
            surface = 'hard'
            if match['surface'] == 1: 
                surface = 'clay'
            elif match['surface'] == 2: 
                surface = 'grass'

            if (number+1) > last_saved:
                tournament_str = match['tournament']
                tournament_str = tournament_str.split('\n')[0]
                time_str = match['time']
                print(f'{number+1}/{number_of_matches}', '\r', end='')
                player1 = match['player1']
                player2 = match['player2']
                tournament_type = match['type']

                file: str = ""

                if "ATP" in tournament_type:
                    file = file1
                elif "WTA" in tournament_type:
                    file = file2
                else:
                    file = file3

                with open(file, 'a') as fileObj:
                    # Write in the new format: "player1" vs "player2"
                    fileObj.write(f'{player1} vs {player2}\n')
                    fileObj.write(f'{surface}\n')
                    match_details = tournament_type + " " + tournament_str + " " + time_str
                    match_details = match_details.replace("\n", " ")
                    fileObj.write(match_details + "\n\n")
                    
                time.sleep(1)
             
    except Exception as e:
        print(f"Error in main: {e}")
    finally:
        driver.quit()

def main():
    atp_elo_site = "https://tennisabstract.com/reports/atp_elo_ratings.html"
    wta_elo_site = "https://tennisabstract.com/reports/wta_elo_ratings.html"
    with ThreadPoolExecutor(max_workers=3) as executor:
        executor.submit(scrape_tennis_elo_rankings, (atp_elo_site,))
        executor.submit(scrape_tennis_elo_rankings, (wta_elo_site,))
        executor.submit(scrape_upcoming_games)

if __name__ == '__main__':
    main()

Fetching data from https://tennisabstract.com/reports/atp_elo_ratings.html...
Fetching data from https://tennisabstract.com/reports/wta_elo_ratings.html...
Successfully retrieved the page: Status code 200
Found 510 player rows
Successfully retrieved the page: Status code 200
Row 162 has fewer cells (12) than headers (14)
Row 291 has fewer cells (12) than headers (14)
Row 384 has fewer cells (12) than headers (14)
Row 459 has fewer cells (12) than headers (14)
Data saved to C:\Users\HP\source\repos\Rehoboam\Rehoboam\Data\Tennis\men_elo_rankings.csv
Found 532 player rows
Row 163 has fewer cells (12) than headers (14)
Row 531 has fewer cells (12) than headers (14)
Data saved to C:\Users\HP\source\repos\Rehoboam\Rehoboam\Data\Tennis\women_elo_rankings.csv


In [4]:
#scrape ATP top 1000 players
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
import time
import csv
import os
from threading import Thread

def setup_driver():
    options = webdriver.ChromeOptions()
    options.add_argument('--disable-notifications')
    options.add_argument('--headless')  # Run in background
    options.add_argument('--disable-gpu')  # Recommended for headless
    options.add_argument('--window-size=1920,1080')  # Set a standard window size
    options.add_argument('--no-sandbox')  # Bypass OS security model
    options.add_argument('--disable-dev-shm-usage')  # Overcome limited resource problems
    
    # Add a realistic user agent
    options.add_argument('--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36')
    
    # Some additional useful options
    options.add_argument('--disable-blink-features=AutomationControlled')  # Hide automation
    options.add_experimental_option('excludeSwitches', ['enable-automation'])  # Hide automation
    options.add_experimental_option('useAutomationExtension', False)  # Hide automation
    
    driver = webdriver.Chrome(options=options)
    
    # Execute JS to modify navigator.webdriver flag
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
    
    return driver

def scrape_tennis_rankings(url):
    driver = setup_driver()
    
    try:
        # Navigate to the ATP rankings page
        driver.get(url)
        
        # Accept cookies if the dialog appears
        try:
            cookie_button = WebDriverWait(driver, 10).until(
                EC.element_to_be_clickable((By.CSS_SELECTOR, "button[id*='onetrust-accept-btn-handler']"))
            )
            cookie_button.click()
            print("Accepted cookies")
        except TimeoutException:
            print("No cookie dialog found or already accepted")

        # Switch to Live Ranking
        try:
            live_ranking_button = WebDriverWait(driver, 10).until(
                EC.element_to_be_clickable((By.CLASS_NAME, "toggle--ranking"))
            )
            live_ranking_button.click()
            print("Switched to Live Ranking")
            time.sleep(2)  # Give time for the page to update
        except Exception as e:
            print(f"Could not switch to Live Ranking: {e}")
        
        time.sleep(3)
        # Click "Show more" button
        try:
            show_more_button = WebDriverWait(driver, 10).until(
                EC.element_to_be_clickable((By.CLASS_NAME, "wclButtonLink--bottomIndent"))
            )
            driver.execute_script("arguments[0].scrollIntoView();", show_more_button)
            time.sleep(0.5) 
            driver.execute_script("arguments[0].click();", show_more_button)
            time.sleep(3)  # Wait for new players to load
        except Exception as e:
            print(f"Error clicking 'Show more' button: {e}")
        
        # At this point, we should have loaded up to 1000 players
        # Now extract the data
        all_players = driver.find_elements(By.CLASS_NAME, "rankingTable__row")
        print(f"Players count: {len(all_players)}")
        
        rankings_data = []
        
        # Extract data for top 1000 players
        for player in all_players:
            try:
                # Extract rank
                rank_element = player.find_element(By.CLASS_NAME, "rankingTable__cell--rank")
                rank = rank_element.text.strip()
                
                # Extract player name
                name_element = player.find_element(By.CLASS_NAME, "rankingTable__cell--player")
                name = name_element.text.strip()
                
                # Extract points
                points_element = player.find_element(By.CLASS_NAME, "rankingTable__cell--points")
                points = points_element.text.strip()
                
                # Add data to list
                rankings_data.append({
                    'Rank': ''.join([x for x in rank if x.isnumeric()]),
                    'Name': name.split('\n')[0],
                    'Points': ''.join([x for x in points if x.isnumeric()])
                })
            except NoSuchElementException as e:
                print(f"Error extracting data from player row: {e}")
                continue
        
        # Save data to CSV
        base = r"C:\Users\HP\source\repos\Rehoboam\Rehoboam\Data\Tennis"
        atp_file = "atp_live_rankings.csv"
        wta_file = "wta_live_rankings.csv"
        csv_filename = os.path.join(base, atp_file)  if 'atp' in url else os.path.join(base, wta_file)

        with open(csv_filename, 'w', newline='', encoding='utf-8') as csvfile:
            fieldnames = ['Rank', 'Name', 'Points']
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            writer.writeheader()
            for player_data in rankings_data:
                writer.writerow(player_data)
        
        print(f"Data saved to {csv_filename}")
        
    finally:
        # Close the browser
        driver.quit()

if __name__ == "__main__":
    atp_site: str = "https://www.tennis24.com/rankings/atp/"
    worker = Thread(target=scrape_tennis_rankings, args=(atp_site,))
    worker.start()
    scrape_tennis_rankings("https://www.tennis24.com/rankings/wta/")
    worker.join()

Accepted cookies
Switched to Live Ranking
Accepted cookies
Switched to Live Ranking
Players count: 2229
Players count: 1538
Data saved to C:\Users\HP\source\repos\Rehoboam\Rehoboam\Data\Tennis\wta_live_rankings.csv
Data saved to C:\Users\HP\source\repos\Rehoboam\Rehoboam\Data\Tennis\atp_live_rankings.csv
